In [1]:
# 定义节点类
class TreeNode:
    def __init__(self, value):
        self.value = value
        self.left = None
        self.right = None

# 创建节点
root = TreeNode('A')
root.left = TreeNode('B')
root.right = TreeNode('C')
root.left.left = TreeNode('D')
root.left.right = TreeNode('E')
root.right.right = TreeNode('F')

# 打印二叉树
def print_tree(node, level=0, label="."):
    indent = "   " * level
    print(f"{indent}{label}: {node.value}")
    if node.left:
        print_tree(node.left, level + 1, "L")
    if node.right:
        print_tree(node.right, level + 1, "R")

print_tree(root)


.: A
   L: B
      L: D
      R: E
   R: C
      R: F


In [5]:
class BTreeNode:
    def __init__(self, t, leaf=False):
        self.t = t  # 最小度数
        self.leaf = leaf  # 是否是叶子节点
        self.keys = []  # 存储键
        self.children = []  # 存储子节点

class BTree:
    def __init__(self, t):
        self.root = BTreeNode(t, True)
        self.t = t  # 最小度数

    def traverse(self, node, level=0):
        if node:
            print(f"{'  ' * level}Node keys: {node.keys}")
            if not node.leaf:
                for child in node.children:
                    self.traverse(child, level + 1)

    def insert(self, key):
        root = self.root
        if len(root.keys) == (2 * self.t - 1):
            new_root = BTreeNode(self.t, False)
            new_root.children.append(self.root)
            self.split_child(new_root, 0)
            self.root = new_root

        self.insert_non_full(self.root, key)

    def insert_non_full(self, node, key):
        i = len(node.keys) - 1
        if node.leaf:
            node.keys.append(None)
            while i >= 0 and key < node.keys[i]:
                node.keys[i + 1] = node.keys[i]
                i -= 1
            node.keys[i + 1] = key
        else:
            while i >= 0 and key < node.keys[i]:
                i -= 1
            i += 1
            if len(node.children[i].keys) == (2 * self.t - 1):
                self.split_child(node, i)
                if key > node.keys[i]:
                    i += 1
            self.insert_non_full(node.children[i], key)

    def split_child(self, parent, index):
        t = self.t
        node = parent.children[index]
        new_node = BTreeNode(t, node.leaf)
        parent.children.insert(index + 1, new_node)
        parent.keys.insert(index, node.keys[t - 1])
        new_node.keys = node.keys[t:(2 * t - 1)]
        node.keys = node.keys[0:(t - 1)]

        if not node.leaf:
            new_node.children = node.children[t:(2 * t)]
            node.children = node.children[0:t]

# 创建 B 树并插入初始键
b_tree = BTree(3)
initial_keys = [10, 20, 5, 6, 15, 30, 25, 35]

for key in initial_keys:
    b_tree.insert(key)

# 插入新的键
insert_key = 40
b_tree.insert(insert_key)

# 遍历并打印 B 树的结构
b_tree.traverse(b_tree.root)


Node keys: [10, 25]
  Node keys: [5, 6]
  Node keys: [15, 20]
  Node keys: [30, 35, 40]


In [1]:
class BTreeNode:
    def __init__(self, t, leaf=False):
        self.t = t  # 最小度数
        self.leaf = leaf  # 是否是叶子节点
        self.keys = []  # 存储键
        self.children = []  # 存储子节点

class BTree:
    def __init__(self, t):
        self.root = BTreeNode(t, True)
        self.t = t  # 最小度数

    def print_tree(self):
        self._print_tree(self.root, "", True)

    def _print_tree(self, node, indent, last):
        if node is not None:
            print(indent, "`- " if last else "|- ", end="")
            print(f"Node({', '.join(map(str, node.keys))})")
            indent += "   " if last else "|  "
            for i, child in enumerate(node.children):
                self._print_tree(child, indent, i == len(node.children) - 1)

    def insert(self, key):
        root = self.root
        if len(root.keys) == (2 * self.t - 1):
            s = BTreeNode(self.t, False)
            self.root = s
            s.children.append(root)
            self.split_child(s, 0)
            self.insert_non_full(s, key)
        else:
            self.insert_non_full(root, key)

    def insert_non_full(self, node, key):
        i = len(node.keys) - 1
        if node.leaf:
            node.keys.append(None)
            while i >= 0 and key < node.keys[i]:
                node.keys[i + 1] = node.keys[i]
                i -= 1
            node.keys[i + 1] = key
        else:
            while i >= 0 and key < node.keys[i]:
                i -= 1
            i += 1
            if len(node.children[i].keys) == (2 * self.t - 1):
                self.split_child(node, i)
                if key > node.keys[i]:
                    i += 1
            self.insert_non_full(node.children[i], key)

    def split_child(self, parent, i):
        t = self.t
        y = parent.children[i]
        z = BTreeNode(t, y.leaf)
        parent.children.insert(i + 1, z)
        parent.keys.insert(i, y.keys[t - 1])
        z.keys = y.keys[t:(2 * t - 1)]
        y.keys = y.keys[0:(t - 1)]
        if not y.leaf:
            z.children = y.children[t:(2 * t)]
            y.children = y.children[0:t]

    def delete(self, key):
        root = self.root
        if root.leaf:
            if key in root.keys:
                root.keys.remove(key)
            if not root.keys:
                self.root = None
            return
        self.delete_key(root, key)
        if len(root.keys) == 0 and not root.leaf:
            self.root = root.children[0]

    def delete_key(self, node, key):
        t = self.t
        index = self.find_key(node, key)
        if index < len(node.keys) and node.keys[index] == key:
            if node.leaf:
                node.keys.pop(index)
            else:
                self.delete_from_non_leaf(node, index)
        else:
            if node.leaf:
                return
            flag = (index == len(node.keys))
            if len(node.children[index].keys) < t:
                self.fill(node, index)
            if flag and index > len(node.keys):
                self.delete_key(node.children[index - 1], key)
            else:
                self.delete_key(node.children[index], key)

    def delete_from_non_leaf(self, node, index):
        t = self.t
        key = node.keys[index]
        if len(node.children[index].keys) >= t:
            pred = self.get_pred(node, index)
            node.keys[index] = pred
            self.delete_key(node.children[index], pred)
        elif len(node.children[index + 1].keys) >= t:
            succ = self.get_succ(node, index)
            node.keys[index] = succ
            self.delete_key(node.children[index + 1], succ)
        else:
            self.merge(node, index)
            self.delete_key(node.children[index], key)

    def get_pred(self, node, index):
        current = node.children[index]
        while not current.leaf:
            current = current.children[-1]
        return current.keys[-1]

    def get_succ(self, node, index):
        current = node.children[index + 1]
        while not current.leaf:
            current = current.children[0]
        return current.keys[0]

    def fill(self, node, index):
        t = self.t
        if index != 0 and len(node.children[index - 1].keys) >= t:
            self.borrow_from_prev(node, index)
        elif index != len(node.keys) and len(node.children[index + 1].keys) >= t:
            self.borrow_from_next(node, index)
        else:
            if index != len(node.keys):
                self.merge(node, index)
            else:
                self.merge(node, index - 1)

    def borrow_from_prev(self, node, index):
        child = node.children[index]
        sibling = node.children[index - 1]
        child.keys.insert(0, node.keys[index - 1])
        if not child.leaf:
            child.children.insert(0, sibling.children.pop())
        node.keys[index - 1] = sibling.keys.pop()

    def borrow_from_next(self, node, index):
        child = node.children[index]
        sibling = node.children[index + 1]
        child.keys.append(node.keys[index])
        if not child.leaf:
            child.children.append(sibling.children.pop(0))
        node.keys[index] = sibling.keys.pop(0)

    def merge(self, node, index):
        t = self.t
        child = node.children[index]
        sibling = node.children[index + 1]
        child.keys.append(node.keys[index])
        child.keys.extend(sibling.keys)
        if not child.leaf:
            child.children.extend(sibling.children)
        node.keys.pop(index)
        node.children.pop(index + 1)

    def find_key(self, node, key):
        i = 0
        while i < len(node.keys) and key > node.keys[i]:
            i += 1
        return i

def main():
    # 创建 B 树
    t = 2
    btree = BTree(t)

    # 插入键
    keys_to_insert = [10, 20, 5, 6, 15, 30, 25, 35]
    for key in keys_to_insert:
        btree.insert(key)

    # 打印树结构
    print("Tree before deletion:")
    btree.print_tree()

    # 删除键（选择一个键进行删除，例如 20）
    btree.delete(20)

    # 打印树结构
    print("\nTree after deletion of 20:")
    btree.print_tree()

if __name__ == "__main__":
    main()


Tree before deletion:
 `- Node(10, 20)
    |- Node(5, 6)
    |- Node(15)
    `- Node(25, 30, 35)

Tree after deletion of 20:
 `- Node(10, 25)
    |- Node(5, 6)
    |- Node(15)
    `- Node(30, 35)


In [2]:
class BTreeNode:
    def __init__(self, t, leaf=False):
        self.t = t  # 最小度数
        self.leaf = leaf  # 是否是叶子节点
        self.keys = []  # 存储键
        self.children = []  # 存储子节点

class BTree:
    def __init__(self, t):
        self.root = BTreeNode(t, True)
        self.t = t  # 最小度数

    def search(self, key):
        return self._search(self.root, key)

    def _search(self, node, key):
        i = 0
        while i < len(node.keys) and key > node.keys[i]:
            i += 1
        if i < len(node.keys) and key == node.keys[i]:
            return (node, i)
        elif node.leaf:
            return None
        else:
            return self._search(node.children[i], key)

    def print_tree(self):
        self._print_tree(self.root, "", True)

    def _print_tree(self, node, indent, last):
        if node is not None:
            print(indent, "`- " if last else "|- ", end="")
            print(f"Node({', '.join(map(str, node.keys))})")
            indent += "   " if last else "|  "
            for i, child in enumerate(node.children):
                self._print_tree(child, indent, i == len(node.children) - 1)

    def insert(self, key):
        root = self.root
        if len(root.keys) == (2 * self.t - 1):
            s = BTreeNode(self.t, False)
            self.root = s
            s.children.append(root)
            self.split_child(s, 0)
            self.insert_non_full(s, key)
        else:
            self.insert_non_full(root, key)

    def insert_non_full(self, node, key):
        i = len(node.keys) - 1
        if node.leaf:
            node.keys.append(None)
            while i >= 0 and key < node.keys[i]:
                node.keys[i + 1] = node.keys[i]
                i -= 1
            node.keys[i + 1] = key
        else:
            while i >= 0 and key < node.keys[i]:
                i -= 1
            i += 1
            if len(node.children[i].keys) == (2 * self.t - 1):
                self.split_child(node, i)
                if key > node.keys[i]:
                    i += 1
            self.insert_non_full(node.children[i], key)

    def split_child(self, parent, i):
        t = self.t
        y = parent.children[i]
        z = BTreeNode(t, y.leaf)
        parent.children.insert(i + 1, z)
        parent.keys.insert(i, y.keys[t - 1])
        z.keys = y.keys[t:(2 * t - 1)]
        y.keys = y.keys[0:(t - 1)]
        if not y.leaf:
            z.children = y.children[t:(2 * t)]
            y.children = y.children[0:t]

    def delete(self, key):
        root = self.root
        if root.leaf:
            if key in root.keys:
                root.keys.remove(key)
            if not root.keys:
                self.root = None
            return
        self.delete_key(root, key)
        if len(root.keys) == 0 and not root.leaf:
            self.root = root.children[0]

    def delete_key(self, node, key):
        t = self.t
        index = self.find_key(node, key)
        if index < len(node.keys) and node.keys[index] == key:
            if node.leaf:
                node.keys.pop(index)
            else:
                self.delete_from_non_leaf(node, index)
        else:
            if node.leaf:
                return
            flag = (index == len(node.keys))
            if len(node.children[index].keys) < t:
                self.fill(node, index)
            if flag and index > len(node.keys):
                self.delete_key(node.children[index - 1], key)
            else:
                self.delete_key(node.children[index], key)

    def delete_from_non_leaf(self, node, index):
        t = self.t
        key = node.keys[index]
        if len(node.children[index].keys) >= t:
            pred = self.get_pred(node, index)
            node.keys[index] = pred
            self.delete_key(node.children[index], pred)
        elif len(node.children[index + 1].keys) >= t:
            succ = self.get_succ(node, index)
            node.keys[index] = succ
            self.delete_key(node.children[index + 1], succ)
        else:
            self.merge(node, index)
            self.delete_key(node.children[index], key)

    def get_pred(self, node, index):
        current = node.children[index]
        while not current.leaf:
            current = current.children[-1]
        return current.keys[-1]

    def get_succ(self, node, index):
        current = node.children[index + 1]
        while not current.leaf:
            current = current.children[0]
        return current.keys[0]

    def fill(self, node, index):
        t = self.t
        if index != 0 and len(node.children[index - 1].keys) >= t:
            self.borrow_from_prev(node, index)
        elif index != len(node.keys) and len(node.children[index + 1].keys) >= t:
            self.borrow_from_next(node, index)
        else:
            if index != len(node.keys):
                self.merge(node, index)
            else:
                self.merge(node, index - 1)

    def borrow_from_prev(self, node, index):
        child = node.children[index]
        sibling = node.children[index - 1]
        child.keys.insert(0, node.keys[index - 1])
        if not child.leaf:
            child.children.insert(0, sibling.children.pop())
        node.keys[index - 1] = sibling.keys.pop()

    def borrow_from_next(self, node, index):
        child = node.children[index]
        sibling = node.children[index + 1]
        child.keys.append(node.keys[index])
        if not child.leaf:
            child.children.append(sibling.children.pop(0))
        node.keys[index] = sibling.keys.pop(0)

    def merge(self, node, index):
        t = self.t
        child = node.children[index]
        sibling = node.children[index + 1]
        child.keys.append(node.keys[index])
        child.keys.extend(sibling.keys)
        if not child.leaf:
            child.children.extend(sibling.children)
        node.keys.pop(index)
        node.children.pop(index + 1)

    def find_key(self, node, key):
        i = 0
        while i < len(node.keys) and key > node.keys[i]:
            i += 1
        return i

def main():
    # 创建 B 树
    t = 2
    btree = BTree(t)

    # 插入键
    keys_to_insert = [10, 20, 5, 6, 15, 30, 25, 35]
    for key in keys_to_insert:
        btree.insert(key)

    # 打印树结构
    print("Tree structure:")
    btree.print_tree()

    # 搜索键（选择一个键进行搜索，例如 15 和 50）
    search_keys = [15, 50]
    for key in search_keys:
        result = btree.search(key)
        if result:
            print(f"\nKey {key} found in node with keys: {result[0].keys} at position {result[1]}")
        else:
            print(f"\nKey {key} not found in the B-Tree")

if __name__ == "__main__":
    main()


Tree structure:
 `- Node(10, 20)
    |- Node(5, 6)
    |- Node(15)
    `- Node(25, 30, 35)

Key 15 found in node with keys: [15] at position 0

Key 50 not found in the B-Tree


In [14]:
class BPlusTreeNode:
    def __init__(self, t, leaf=False):
        self.t = t  # 阶数
        self.leaf = leaf  # 是否是叶子节点
        self.keys = []  # 存储键
        self.children = []  # 存储子节点
        self.next = None  # 指向下一个叶子节点的指针

class BPlusTree:
    def __init__(self, t):
        self.root = BPlusTreeNode(t, True)  # 初始化一个空树
        self.t = t

    def insert(self, key):
        root = self.root
        if len(root.keys) == (2 * self.t - 1):  # 根节点满了
            s = BPlusTreeNode(self.t, False)  # 新的根节点
            self.root = s
            s.children.append(root)
            self.split_child(s, 0)
            self.insert_non_full(s, key)
        else:
            self.insert_non_full(root, key)

    def insert_non_full(self, node, key):
        if node.leaf:
            node.keys.append(None)  # 为新键腾出空间
            i = len(node.keys) - 2
            while i >= 0 and key < node.keys[i]:
                node.keys[i + 1] = node.keys[i]
                i -= 1
            node.keys[i + 1] = key
            node.keys.sort()
        else:
            i = len(node.keys) - 1
            while i >= 0 and key < node.keys[i]:
                i -= 1
            i += 1
            if len(node.children[i].keys) == (2 * self.t - 1):
                self.split_child(node, i)
                if key > node.keys[i]:
                    i += 1
            self.insert_non_full(node.children[i], key)

    def split_child(self, parent, i):
        t = self.t
        node = parent.children[i]
        new_node = BPlusTreeNode(t, node.leaf)
        parent.children.insert(i + 1, new_node)
        parent.keys.insert(i, node.keys[t - 1])
        new_node.keys = node.keys[t:]
        node.keys = node.keys[:t - 1]
        if not node.leaf:
            new_node.children = node.children[t:]
            node.children = node.children[:t]

        if node.leaf:
            new_node.next = node.next
            node.next = new_node

    def display(self, node=None, level=0, pos='Root'):
        if node is None:
            node = self.root
        indent = "  " * level
        if node.leaf:
            print(f"{indent}{pos} Leaf Node: {node.keys}")
        else:
            print(f"{indent}{pos} Internal Node: {node.keys}")
            for i, child in enumerate(node.children):
                self.display(child, level + 1, pos=f'Child {i}')

# Example usage:
bpt = BPlusTree(t=2)
keys = [101, 102, 103, 104, 105]

for key in keys:
    bpt.insert(key)

bpt.display()


Root Internal Node: [102]
  Child 0 Leaf Node: [101]
  Child 1 Leaf Node: [103, 104, 105]


In [17]:
import time
import random

# B树节点类
class BTreeNode:
    def __init__(self, t, leaf=False):
        self.t = t  # 阶数
        self.leaf = leaf  # 是否是叶子节点
        self.keys = []  # 存储键
        self.children = []  # 存储子节点

class BTree:
    def __init__(self, t):
        self.root = BTreeNode(t, True)  # 初始化一个空树
        self.t = t

    def search(self, key, node=None):
        if node is None:
            node = self.root
        i = 0
        while i < len(node.keys) and key > node.keys[i]:
            i += 1
        if i < len(node.keys) and key == node.keys[i]:
            return True
        if node.leaf:
            return False
        return self.search(key, node.children[i])

    def insert(self, key):
        root = self.root
        if len(root.keys) == (2 * self.t - 1):  # 根节点满了
            s = BTreeNode(self.t, False)  # 新的根节点
            self.root = s
            s.children.append(root)
            self.split_child(s, 0)
            self.insert_non_full(s, key)
        else:
            self.insert_non_full(root, key)

    def insert_non_full(self, node, key):
        if node.leaf:
            node.keys.append(None)  # 为新键腾出空间
            i = len(node.keys) - 2
            while i >= 0 and key < node.keys[i]:
                node.keys[i + 1] = node.keys[i]
                i -= 1
            node.keys[i + 1] = key
        else:
            i = len(node.keys) - 1
            while i >= 0 and key < node.keys[i]:
                i -= 1
            i += 1
            if len(node.children[i].keys) == (2 * self.t - 1):
                self.split_child(node, i)
                if key > node.keys[i]:
                    i += 1
            self.insert_non_full(node.children[i], key)

    def split_child(self, parent, i):
        t = self.t
        node = parent.children[i]
        new_node = BTreeNode(t, node.leaf)
        parent.children.insert(i + 1, new_node)
        parent.keys.insert(i, node.keys[t - 1])
        new_node.keys = node.keys[t:]
        node.keys = node.keys[:t - 1]
        if not node.leaf:
            new_node.children = node.children[t:]
            node.children = node.children[:t]

# B+树节点类
class BPlusTreeNode:
    def __init__(self, t, leaf=False):
        self.t = t  # 阶数
        self.leaf = leaf  # 是否是叶子节点
        self.keys = []  # 存储键
        self.children = []  # 存储子节点
        self.next = None  # 指向下一个叶子节点的指针

class BPlusTree:
    def __init__(self, t):
        self.root = BPlusTreeNode(t, True)  # 初始化一个空树
        self.t = t

    def search(self, key, node=None):
        if node is None:
            node = self.root
        while node:
            i = 0
            while i < len(node.keys) and key > node.keys[i]:
                i += 1
            if i < len(node.keys) and key == node.keys[i]:
                return True
            if node.leaf:
                return False
            node = node.children[i]
        return False

    def insert(self, key):
        root = self.root
        if len(root.keys) == (2 * self.t - 1):  # 根节点满了
            s = BPlusTreeNode(self.t, False)  # 新的根节点
            self.root = s
            s.children.append(root)
            self.split_child(s, 0)
            self.insert_non_full(s, key)
        else:
            self.insert_non_full(root, key)

    def insert_non_full(self, node, key):
        if node.leaf:
            node.keys.append(None)  # 为新键腾出空间
            i = len(node.keys) - 2
            while i >= 0 and key < node.keys[i]:
                node.keys[i + 1] = node.keys[i]
                i -= 1
            node.keys[i + 1] = key
            node.keys.sort()
        else:
            i = len(node.keys) - 1
            while i >= 0 and key < node.keys[i]:
                i -= 1
            i += 1
            if len(node.children[i].keys) == (2 * self.t - 1):
                self.split_child(node, i)
                if key > node.keys[i]:
                    i += 1
            self.insert_non_full(node.children[i], key)

    def split_child(self, parent, i):
        t = self.t
        node = parent.children[i]
        new_node = BPlusTreeNode(t, node.leaf)
        parent.children.insert(i + 1, new_node)
        parent.keys.insert(i, node.keys[t - 1])
        new_node.keys = node.keys[t:]
        node.keys = node.keys[:t - 1]
        if not node.leaf:
            new_node.children = node.children[t:]
            node.children = node.children[:t]

        if node.leaf:
            new_node.next = node.next
            node.next = new_node

# 测试代码
def measure_search_time(tree, keys, search_key):
    start_time = time.time()
    tree.search(search_key)
    end_time = time.time()
    search_time = end_time - start_time
    return search_time

# 创建 B 树 和 B+ 树
b_tree = BTree(t=2)
b_plus_tree = BPlusTree(t=2)

# 生成10000个随机数据
keys = random.sample(range(1, 20001), 10000)

# 插入相同的数据
for key in keys:
    b_tree.insert(key)
    b_plus_tree.insert(key)

# 测量 B 树 和 B+ 树 的查找时间
search_key = keys[random.randint(0, 9999)]  # 随机选择一个查找的键
b_tree_time = measure_search_time(b_tree, keys, search_key)
b_plus_tree_time = measure_search_time(b_plus_tree, keys, search_key)

print(f"B 树 查找时间: {b_tree_time:.6f} 秒")
print(f"B+ 树 查找时间: {b_plus_tree_time:.6f} 秒")


B 树 查找时间: 0.000000 秒
B+ 树 查找时间: 0.000000 秒
